# Unidade II — Análise de Dados

## Similaridade e dissimilaridade

**Carga estimada:** 3 horas e 30 minutos  
**Pré-requisitos:** tipos de atributos, somatórios, vetores e raiz quadrada.

> **Pergunta norteadora:** quando dois objetos devem ser considerados próximos, e como a representação dos atributos altera essa resposta?


## Objetivos de aprendizagem

Ao concluir este notebook, você será capaz de:

- distinguir matriz de dados e matriz de dissimilaridades;
- calcular distâncias Euclidiana, Manhattan e Minkowski;
- calcular similaridade do cosseno e de Jaccard;
- escolher medidas para atributos numéricos, binários e mistos;
- explicar por que escala, codificação e dimensionalidade alteram vizinhanças.


## Proximidade e representação

Uma **similaridade** cresce quando objetos se parecem; uma **dissimilaridade** ou distância cresce quando diferem. A matriz de dados $\mathbf{X}$ possui objetos nas linhas e atributos nas colunas. Uma matriz de dissimilaridades $\mathbf{D}$ compara cada par de objetos, é quadrada e, para distâncias simétricas, satisfaz $D_{ij}=D_{ji}$ e $D_{ii}=0$.

Não existe medida universal. A escolha expressa o que significa estar próximo no problema e precisa respeitar a natureza dos atributos.


## Distâncias para atributos numéricos

Para $\mathbf{x},\mathbf{y}\in\mathbb{R}^d$, a distância de Minkowski de ordem $p\geq 1$ é

$$
d_p(\mathbf{x},\mathbf{y})=\left(\sum_{j=1}^{d}|x_j-y_j|^p\right)^{1/p}.
$$

Quando $p=1$, obtemos Manhattan; quando $p=2$, Euclidiana. Manhattan soma diferenças absolutas. Euclidiana enfatiza diferenças grandes por elevar ao quadrado antes da raiz.

Considere $\mathbf{x}=(1,2)$ e $\mathbf{y}=(4,6)$:

$$d_1=|1-4|+|2-6|=7,$$

$$d_2=\sqrt{(1-4)^2+(2-6)^2}=5.$$


In [1]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist, cosine, jaccard
from sklearn.preprocessing import StandardScaler

x = np.array([1.0, 2.0])
y = np.array([4.0, 6.0])

manhattan = np.abs(x - y).sum()
euclidiana = np.sqrt(np.sum((x - y) ** 2))
pd.Series({"Manhattan": manhattan, "Euclidiana": euclidiana})


Manhattan     7.0
Euclidiana    5.0
dtype: float64

### Matriz de dissimilaridades

Vamos calcular todas as distâncias entre quatro objetos. A diagonal deve ser zero e a matriz, simétrica.


In [2]:
objetos = pd.DataFrame(
    {"atributo_1": [1, 2, 5, 8], "atributo_2": [2, 2, 6, 9]},
    index=["A", "B", "C", "D"],
)
matriz_d = pd.DataFrame(
    cdist(objetos, objetos, metric="euclidean"),
    index=objetos.index, columns=objetos.index,
).round(2)
matriz_d


,A,B,C,D
A,0.00,1.00,5.66,9.90
B,1.00,0.00,5.00,9.22
C,5.66,5.00,0.00,4.24
D,9.90,9.22,4.24,0.00


## O efeito da escala

Se renda varia em milhares e idade em dezenas, a renda pode dominar uma distância, mesmo quando não deveria dominar o conceito de semelhança. Padronizar transforma cada atributo numérico pela média e desvio-padrão:

$$z_{ij}=\frac{x_{ij}-\bar{x}_j}{s_j}.$$

A transformação não é automaticamente correta: ela atribui escala comparável, mas a relevância substantiva de cada atributo ainda precisa ser decidida.


In [3]:
pessoas = pd.DataFrame(
    {"idade": [20, 22, 55], "renda_mensal": [2000, 8000, 8500]},
    index=["P1", "P2", "P3"],
)
sem_escala = pd.DataFrame(cdist(pessoas, pessoas), index=pessoas.index, columns=pessoas.index)
padronizados = StandardScaler().fit_transform(pessoas)
com_escala = pd.DataFrame(cdist(padronizados, padronizados), index=pessoas.index, columns=pessoas.index)

pd.concat({"sem padronização": sem_escala, "padronizada": com_escala}).round(2)


P1       P2       P3
sem padronização P1     0.00  6000.00  6500.09
                 P2  6000.00     0.00   501.09
                 P3  6500.09   501.09     0.00
padronizada      P1     0.00     2.04     3.10
                 P2     2.04     0.00     2.06
                 P3     3.10     2.06     0.00

## Similaridade do cosseno

Para vetores não nulos,

$$
\operatorname{sim}_{\cos}(\mathbf{x},\mathbf{y})=\frac{\mathbf{x}^T\mathbf{y}}{\|\mathbf{x}\|_2\|\mathbf{y}\|_2}.
$$

Ela mede o ângulo, sendo útil quando a direção do perfil importa mais que a magnitude, como em vetores de frequência de termos. Vetores proporcionais têm cosseno 1, ainda que suas magnitudes sejam diferentes.


In [4]:
documento_a = np.array([2.0, 1.0, 0.0])
documento_b = np.array([4.0, 2.0, 0.0])
documento_c = np.array([0.0, 1.0, 3.0])

pd.Series({
    "sim(A, B)": 1 - cosine(documento_a, documento_b),
    "sim(A, C)": 1 - cosine(documento_a, documento_c),
}).round(3)


sim(A, B)    1.000
sim(A, C)    0.141
dtype: float64

## Atributos binários e Jaccard

Para conjuntos $A$ e $B$, a similaridade de Jaccard é

$$J(A,B)=\frac{|A\cap B|}{|A\cup B|}.$$

Em vetores binários assimétricos, zeros conjuntos não contam: a ausência simultânea de milhares de itens não deveria tornar duas cestas artificialmente semelhantes. A distância é $1-J$.


In [5]:
cesta_a = np.array([1, 1, 0, 0, 1], dtype=bool)
cesta_b = np.array([1, 0, 1, 0, 1], dtype=bool)
intersecao = np.logical_and(cesta_a, cesta_b).sum()
uniao = np.logical_or(cesta_a, cesta_b).sum()

pd.Series({
    "Jaccard manual": intersecao / uniao,
    "Jaccard via SciPy": 1 - jaccard(cesta_a, cesta_b),
}).round(3)


Jaccard manual       0.5
Jaccard via SciPy    0.5
dtype: float64

## Dissimilaridade de Gower para atributos mistos

Uma base pode descrever a mesma pessoa por idade numérica, escolaridade ordinal, município nominal e preferências binárias. Aplicar Euclidiana diretamente exigiria converter categorias em números e criaria ordens ou distâncias sem significado. A proposta de Gower é calcular uma dissimilaridade apropriada para **cada atributo**, levar as contribuições para uma escala comum entre 0 e 1 e então obter uma média ponderada.

Para os objetos $i$ e $k$, a dissimilaridade agregada pode ser escrita como

$$
d_G(i,k)=
\frac{\sum_{j=1}^{p} a_j r_{ijk}\,\delta_{ijk}}
{\sum_{j=1}^{p} a_j r_{ijk}}.
$$

Nessa expressão, $\delta_{ijk}\in[0,1]$ é a diferença parcial no atributo $j$; $a_j\geq0$ é seu peso; e $r_{ijk}$ vale 1 quando o atributo participa da comparação e 0 quando deve ser ignorado, por exemplo devido a valor ausente. Se o denominador for zero, o par não possui informação comparável e a dissimilaridade deve ser considerada indefinida.

| Tipo do atributo | Dissimilaridade parcial $\delta_{ijk}$ |
|---|---|
| Numérico | $\lvert x_{ij}-x_{kj} \rvert/R_j$, em que $R_j$ é a amplitude observada do atributo |
| Nominal | 0 quando os valores são iguais e 1 quando são diferentes |
| Ordinal | Substituir a categoria por sua posição normalizada em $[0,1]$ e aplicar a diferença absoluta |
| Binário simétrico | 0 para igualdade e 1 para diferença; zeros conjuntos participam |
| Binário assimétrico | 0 para duas presenças, 1 para presença contra ausência; duas ausências são retiradas daquela comparação |

O resultado também fica entre 0 e 1 quando os pesos são não negativos: valores próximos de 0 indicam maior semelhança segundo as definições adotadas. Isso não torna a medida automática: tipos, ordens, pesos, amplitude e tratamento de ausências continuam sendo decisões do problema.


### Exemplo com perfis de clientes

Vamos comparar três perfis. `idade` e `renda` são numéricos; `escolaridade` é ordinal; `cidade` é nominal; `compra_online` e `assinante_premium` são binários assimétricos, pois uma presença compartilhada é evidência de semelhança, mas duas ausências não são.


In [6]:
perfis = pd.DataFrame(
    {
        "idade": [25, 28, 55],
        "renda": [3000, 3300, 8000],
        "escolaridade": ["Médio", "Superior", "Fundamental"],
        "cidade": ["Recife", "Recife", "Olinda"],
        "compra_online": [1, 1, 0],
        "assinante_premium": [0, 0, 1],
    },
    index=["Ana", "Bruno", "Carla"],
)
perfis


,idade,renda,escolaridade,cidade,compra_online,assinante_premium
Ana,25,3000,Médio,Recife,1,0
Bruno,28,3300,Superior,Recife,1,0
Carla,55,8000,Fundamental,Olinda,0,1


In [7]:
def matriz_gower(
    dados: pd.DataFrame,
    numericos: list[str],
    nominais: list[str],
    ordens: dict[str, list[str]],
    binarios_assimetricos: list[str],
) -> pd.DataFrame:
    """Calcula Gower didático com pesos iguais e ausências ignoradas por par."""
    amplitudes = dados[numericos].max() - dados[numericos].min()
    resultado = np.zeros((len(dados), len(dados)), dtype=float)

    for i in range(len(dados)):
        for k in range(i + 1, len(dados)):
            contribuicoes: list[float] = []

            for coluna in numericos:
                valores = dados.iloc[[i, k]][coluna]
                if valores.notna().all():
                    amplitude = amplitudes[coluna]
                    delta = (
                        0.0
                        if amplitude == 0
                        else abs(valores.iloc[0] - valores.iloc[1]) / amplitude
                    )
                    contribuicoes.append(float(delta))

            for coluna in nominais:
                valores = dados.iloc[[i, k]][coluna]
                if valores.notna().all():
                    contribuicoes.append(float(valores.iloc[0] != valores.iloc[1]))

            for coluna, categorias in ordens.items():
                valores = dados.iloc[[i, k]][coluna]
                if valores.notna().all():
                    divisor = max(len(categorias) - 1, 1)
                    posicao = {
                        categoria: ordem / divisor
                        for ordem, categoria in enumerate(categorias)
                    }
                    diferenca = abs(
                        posicao[valores.iloc[0]] - posicao[valores.iloc[1]]
                    )
                    contribuicoes.append(diferenca)

            for coluna in binarios_assimetricos:
                valores = dados.iloc[[i, k]][coluna]
                if valores.notna().all() and not (valores == 0).all():
                    contribuicoes.append(float(valores.iloc[0] != valores.iloc[1]))

            distancia = np.mean(contribuicoes) if contribuicoes else np.nan
            resultado[i, k] = resultado[k, i] = distancia

    return pd.DataFrame(resultado, index=dados.index, columns=dados.index)


matriz_g = matriz_gower(
    perfis,
    numericos=["idade", "renda"],
    nominais=["cidade"],
    ordens={"escolaridade": ["Fundamental", "Médio", "Superior"]},
    binarios_assimetricos=["compra_online", "assinante_premium"],
)
matriz_g.round(3)


,Ana,Bruno,Carla
Ana,0.000,0.132,0.917
Bruno,0.132,0.000,0.973
Carla,0.917,0.973,0.000


Ana e Bruno formam o par mais próximo. Suas contribuições são: idade $3/30=0{,}10$; renda $300/5000=0{,}06$; escolaridade $0{,}50$; cidade 0; e compra on-line 0. `assinante_premium` não entra no denominador porque ambos possuem zero nesse atributo assimétrico. Logo,

$$d_G(\text{Ana},\text{Bruno})=\frac{0{,}10+0{,}06+0{,}50+0+0}{5}=0{,}132.$$

Carla difere mais nos atributos numéricos e categóricos, por isso suas dissimilaridades são maiores. A matriz é simétrica e tem diagonal zero.

> **Limitações:** a amplitude numérica pode ser distorcida por outliers; pesos iguais podem supervalorizar blocos com muitos atributos semelhantes; ordens precisam ser substantivamente defensáveis; e pares com ausências podem ser comparados usando conjuntos diferentes de atributos. Em um fluxo preditivo, amplitudes, categorias e outras decisões orientadas pelos dados devem ser aprendidas apenas no conjunto de treino.


## Atributos mistos e alta dimensionalidade

Dados reais combinam números, categorias, ordens e indicadores binários. Converter categorias arbitrariamente em inteiros e aplicar Euclidiana cria ordens e intervalos falsos. Uma alternativa é calcular contribuições adequadas por atributo, normalizá-las e agregá-las, como na dissimilaridade de Gower. Valores ausentes e pesos também precisam ser tratados explicitamente.

Em muitas dimensões, distâncias podem se concentrar: o vizinho mais próximo e o mais distante tornam-se relativamente parecidos. Seleção de atributos, redução de dimensionalidade e medidas específicas do domínio podem ser necessárias.

> **U02-NB03-V01 — Verifique seu entendimento:** duas cestas compartilham 2 itens e possuem 5 itens distintos na união. Qual é a similaridade de Jaccard e qual é a distância correspondente?

> **U02-NB03-E01 — Exercício:** crie quatro perfis com idade, renda, escolaridade ordinal e três preferências binárias. Proponha uma medida de dissimilaridade, explicando como cada tipo será tratado, se haverá padronização e que significado terá uma distância pequena. Compare ao menos dois pares manualmente.


## Como escolher uma medida de proximidade

Escolher uma medida significa formalizar o que **parecido** quer dizer no problema. A decisão não deve ser tomada apenas porque uma função está disponível na biblioteca: medidas diferentes podem produzir vizinhos, grupos e casos atípicos diferentes sobre os mesmos registros.

| Medida | Quando costuma ser adequada | Exemplo de aplicação | Cuidados principais |
|---|---|---|---|
| Euclidiana | Atributos numéricos em que deslocamentos geométricos e magnitude são relevantes | Distância entre objetos descritos por medidas físicas comparáveis | Escalas maiores podem dominar; diferenças grandes em uma coordenada recebem maior ênfase; outliers podem alterar fortemente as vizinhanças |
| Manhattan | Atributos numéricos quando as diferenças devem ser acumuladas coordenada a coordenada | Comparação de perfis por desvios absolutos | Continua dependente da escala; é menos enfática que a Euclidiana diante de uma diferença muito grande, mas não é imune a outliers |
| Minkowski | Quando há justificativa para controlar, por meio de $p$, quanto diferenças grandes influenciam a distância | Generalização de Manhattan ($p=1$) e Euclidiana ($p=2$) | Para ser uma métrica, requer $p\geq1$; $p$ não deve ser escolhido arbitrariamente ou ajustado com o conjunto de teste |
| Cosseno | Vetores em que a direção do perfil importa mais que sua magnitude | Documentos representados por frequências ou pesos de termos | Não é definido para vetor nulo; vetores proporcionais podem ter similaridade 1 mesmo com magnitudes muito diferentes; valores negativos exigem interpretação cuidadosa |
| Jaccard | Conjuntos ou atributos binários assimétricos, quando presenças compartilhadas são informativas | Itens de uma cesta, sintomas presentes ou interesses declarados | Ausências conjuntas são ignoradas; se os dois conjuntos forem vazios, a união será vazia e uma convenção deverá ser declarada |
| Gower | Registros com atributos numéricos, nominais, ordinais e binários misturados | Perfis de clientes contendo idade, escolaridade, município e preferências | Cada tipo precisa de uma contribuição coerente; normalização, pesos e valores ausentes devem ser tratados explicitamente |

### Roteiro de decisão

1. **Defina a unidade de análise e a finalidade.** Proximidade para recuperar documentos pode significar algo diferente de proximidade para detectar uma transação incomum.
2. **Classifique os atributos.** Verifique quais são numéricos, nominais, ordinais ou binários e evite criar distâncias artificiais com códigos de categorias.
3. **Decida o que deve permanecer invariável.** Se multiplicar todo o perfil por uma constante não deveria alterar a semelhança, o cosseno pode ser mais coerente que uma distância de magnitude.
4. **Pergunte se ausências conjuntas representam semelhança.** Quando milhares de itens não adquiridos em comum não são evidência, Jaccard é preferível a uma comparação binária simétrica.
5. **Examine escalas, dispersões e pesos.** Padronizar impede domínio puramente numérico, mas não decide a relevância substantiva de cada atributo.
6. **Defina casos especiais.** Documente o tratamento de valores ausentes, vetores nulos, conjuntos vazios, categorias novas e atributos constantes.
7. **Compare os resultados.** Inspecione pares e vizinhanças conhecidos, faça análise de sensibilidade e valide a escolha na tarefa posterior sem usar o conjunto de teste para decidir a configuração.

> **Regra prática:** o tipo de dado restringe as opções, mas o significado da proximidade e a finalidade da análise determinam a escolha final. Não existe uma medida universalmente melhor.


## Síntese

- A proximidade depende da representação, do tipo dos atributos e do problema.
- Manhattan e Euclidiana são casos da distância de Minkowski.
- Escalas diferentes podem dominar distâncias numéricas.
- Cosseno compara direção; Jaccard compara presença compartilhada.
- Atributos mistos exigem contribuições compatíveis, não códigos numéricos arbitrários.

## Referências

- HAN, Jiawei; PEI, Jian; TONG, Hanghang. *Data Mining: Concepts and Techniques*. 4. ed. Cambridge: Morgan Kaufmann/Elsevier, 2023. Cap. 2, seção 2.3.
- GOWER, J. C. A general coefficient of similarity and some of its properties. *Biometrics*, v. 27, n. 4, p. 857–871, 1971.
